# PRDetect — Pipeline GATv2 Chuẩn Dự Án & Test Trên RAID

Notebook này tái sử dụng **100% mã nguồn có sẵn trong dự án**:
- Xây dựng đồ thị: Sử dụng trực tiếp `building_graph.py`
- Mô hình: Sử dụng `model/GATv2.py` (Kế thừa và thay thế `model/GCN2.py`)
- Huấn luyện: Sử dụng `train_gatv2.py` (Cấu hình chuẩn `gcn.py`: seed 2024, lr 0.0001)
- Đánh giá: Sử dụng `test_gatv2.py` để test trên **HC3 Test** và **Tập RAID** (`raid_llama_test`, `raid_mistral_test`)

> **Lưu ý trên Kaggle**: Chọn **GPU T4** và bật **Internet: ON**.


## Bước 1: Cài đặt thư viện phụ thuộc


In [ ]:
!pip install torch-geometric transformers datasets scikit-learn tqdm -q
!python -m spacy download en_core_web_sm -q
print("Cai dat moi truong hoan tat!")


## Bước 2: Kiểm tra cấu trúc thư mục dự án
Đảm bảo đã có đầy đủ mã nguồn và các file trong `original_text/`:
- `original_text/hc3_train.json`
- `original_text/hc3_val.json`
- `original_text/hc3_test.json`
- `original_text/raid_llama_test.json`
- `original_text/raid_mistral_test.json`


In [ ]:
import os

required_files = [
    "building_graph.py",
    "train_gatv2.py",
    "test_gatv2.py",
    "model/GATv2.py",
    "original_text/hc3_train.json",
    "original_text/hc3_val.json",
    "original_text/hc3_test.json",
    "original_text/raid_llama_test.json",
    "original_text/raid_mistral_test.json"
]

print("=== KIEM TRA FILE DU AN ===")
missing = [f for f in required_files if not os.path.exists(f)]
if missing:
    print(f"[CANH BAO] Thieu cac file sau: {missing}")
else:
    print("[OK] Day du tat ca cac file ma nguon va du lieu goc!")


## Bước 3: Xây dựng Đồ thị Cú pháp bằng `building_graph.py`
Gọi trực tiếp script `building_graph.py` từ dự án để sinh ra các file `.pkl` trong `graph_data/`:
1. `hc3_train.pkl` (8,000 mẫu)
2. `hc3_val.pkl` (1,000 mẫu)
3. `hc3_test.pkl` (1,000 mẫu)
4. `raid_llama_test.pkl` (2,000 mẫu)
5. `raid_mistral_test.pkl` (2,000 mẫu)


In [ ]:
# Chay truc tiep building_graph.py co san cua nhom
!python building_graph.py -f hc3_train hc3_val hc3_test raid_llama_test raid_mistral_test


## Bước 4: Huấn luyện GATv2 bằng `train_gatv2.py`
Chạy file `train_gatv2.py` của dự án với các siêu tham số đồng nhất với `gcn.py`:
- `dataset`: `hc3` (đọc `graph_data/hc3_train.pkl` và `hc3_val.pkl`)
- `seed`: `2024`
- `lr`: `0.0001`
- `epochs`: `40`
- `patience`: `5` (tăng lên 5 để GATv2 có thêm cơ hội hội tụ tối ưu)
- `heads`: `4`


In [ ]:
# Chay script huan luyen GATv2 chuan cua du an
!python train_gatv2.py --dataset hc3 --seed 2024 --epochs 40 --lr 0.0001 --patience 5 --heads 4


## Bước 5: Đánh giá trên HC3 Test (In-Domain)
Sử dụng `test_gatv2.py` để test tập `hc3_test.pkl` và ghi kết quả vào `result/test_result_gatv2.txt`:


In [ ]:
# Test tren HC3 Test
!python test_gatv2.py --file hc3_test --dataset hc3 --seed 2024 -s


## Bước 6: Đánh giá trên tập RAID (Cross-Domain / LLM Thử nghiệm)
Thực hiện đánh giá khả năng tổng quát hóa trên tập dữ liệu **RAID**:
1. `raid_llama_test` (Văn bản sinh bởi LLaMA)
2. `raid_mistral_test` (Văn bản sinh bởi Mistral)


In [ ]:
print("=== [1/2] TEST TREN RAID - LLAMA ===")
!python test_gatv2.py --file raid_llama_test --dataset hc3 --seed 2024 -s

print("\n=== [2/2] TEST TREN RAID - MISTRAL ===")
!python test_gatv2.py --file raid_mistral_test --dataset hc3 --seed 2024 -s


## Bước 7: Tổng hợp kết quả Kiểm thử
Đọc và hiển thị toàn bộ kết quả đã lưu trong `result/test_result_gatv2.txt`:


In [ ]:
result_file = "./result/test_result_gatv2.txt"
if os.path.exists(result_file):
    print("=== BANG KET QUA TEST GATv2 ===")
    with open(result_file, "r", encoding="utf-8") as f:
        print(f.read())
else:
    print(f"Chua tim thay file {result_file}")
